# Predicting the Presence of Heart Disease from Clinical Data
### Final Project – CPE393: Introduction to Data Science with Python (Summer 2026, KMUTT)

- **Group:** IUTO
- **Members:** Kylian RIBEROU (59), Leni CHABILAN (35), Romain MECHAIN (52)
- **GitHub repo:** https://github.com/LeniChabilan/CPE_Project_Group_IUTO

---
This notebook walks through a full data science workflow on a heart disease dataset: we define the problem, describe and clean the data, explore it, engineer a few features, train and compare several machine learning models, evaluate them, and finish with an honest discussion of responsible-AI concerns.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, classification_report, roc_curve, auc)

import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42          # fixed seed
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

## 2. Problem Definition

Our goal is a binary classifier that predicts whether a patient has heart disease (`target = 1`) or not (`target = 0`), using 13 clinical variables collected during a fairly standard check-up: age, sex, blood pressure, cholesterol, ECG results, maximum heart rate, and so on.

Why this problematic ? Cardiovascular disease is one of the leading causes of death worldwide, and even a simple tool that flags at-risk patients from easily-measured data could help push someone find a doctor and get deepers diagnosis.

So the question is: *from standard clinical measurements, can we reliably predict heart disease, and which variables carry the most signal?*

This notebooks conducts a supervised classification task, with the goal of training an accurate and precise model.9*

## 3. Dataset Description

- **Source:** the Heart Disease Dataset, a version of the classic *UCI Heart Disease / Cleveland* database that circulates widely on Kaggle as `heart.csv` (e.g. the "Heart Disease Dataset" by johnsmith88).
- **Typical Kaggle URL:** https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset
- **License / usage:** public dataset, commonly used for teaching and research (derived from the UCI Machine Learning Repository, public domain for academic use).
- **File format:** CSV (`heart.csv`)
- **Rows (before cleaning):** 1025
- **Columns:** 14 (13 features + 1 target)
- **Target:** `target` (0 = no heart disease, 1 = heart disease present)

### Variable dictionary

| Variable | Description | Type |
|---|---|---|
| age | Patient age (years) | Numeric |
| sex | Sex (1 = male, 0 = female) | Binary categorical |
| cp | Chest pain type (0-3) | Categorical |
| trestbps | Resting blood pressure (mm Hg) | Numeric |
| chol | Serum cholesterol (mg/dl) | Numeric |
| fbs | Fasting blood sugar > 120 mg/dl (1 = true, 0 = false) | Binary categorical |
| restecg | Resting ECG results (0-2) | Categorical |
| thalach | Maximum heart rate achieved | Numeric |
| exang | Exercise-induced angina (1 = yes, 0 = no) | Binary categorical |
| oldpeak | ST depression induced by exercise | Numeric |
| slope | Slope of the peak exercise ST segment (0-2) | Categorical |
| ca | Number of major vessels colored by fluoroscopy (valid: 0-3) | Categorical |
| thal | Thalassemia test result (valid: 1-3) | Categorical |
| target | Heart disease present (1) or absent (0) | Binary target |

Before all, we load the data and see what it looks like

In [ ]:
df = pd.read_csv('data/heart_diseases/heart.csv')
print(f"Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

A few things worth keeping in mind (we'll come back to them in the responsible-AI section):

- This is a heavily resampled/copied version of the original Cleveland database and contains a lot of duplicate rows (more on that during cleaning), so we should be careful about treating it as representative of a real patient population.
- It doesn't tell us the geographic or demographic origin of the patients, which limits how far the model can generalize.
- A couple of categorical columns contain codes outside the documented range (`ca` and `thal` — see below), which points to data-entry or encoding errors.

## 4. Preprocessing and Cleaning

We'll go through this in order: missing values, duplicates, data types, out-of-codebook values, and finally outliers.

### 4.1 Missing values

In [ ]:
missing = df.isnull().sum()
print(missing)
print(f"\nTotal missing values: {missing.sum()}")

No missing values anywhere across the 14 columns, so there's nothing to impute here.

### 4.2 Duplicates

In [ ]:
n_dup = df.duplicated().sum()
print(f"Exact duplicate rows: {n_dup} out of {len(df)} ({n_dup/len(df)*100:.1f}%)")

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
print(f"Shape after dropping duplicates: {df.shape[0]} rows x {df.shape[1]} columns")

This is a big one: 723 rows — roughly 70% of the file — are exact duplicates. That confirms the Kaggle version was built by duplicating/resampling the original Cleveland data (which only has around 300 unique patients).

We drop them, and not just for tidiness. Leaving duplicates in would let the same patient show up in both the train and test sets, which leaks information and inflates the scores. After removing them we're left with 302 unique patients, right in line with the 303 in the original Cleveland database. The reference Kaggle notebook we looked at didn't catch this — it trained on the duplicated data, which goes a long way toward explaining its ~92% accuracy. Classic data leakage.

### 4.3 Data types and category-code consistency

In [ ]:
print(df.dtypes)

In [ ]:
print("Unique values of 'ca' (expected 0-3):", sorted(df['ca'].unique()))
print("Unique values of 'thal' (expected 1-3):", sorted(df['thal'].unique()))

In [ ]:
invalid_ca = (df['ca'] == 4).sum()
invalid_thal = (df['thal'] == 0).sum()
print(f"Rows with ca=4 (out of codebook): {invalid_ca}")
print(f"Rows with thal=0 (out of codebook): {invalid_thal}")

In [ ]:
df = df[(df['ca'] != 4) & (df['thal'] != 0)].reset_index(drop=True)
print(f"Shape after removing invalid codes: {df.shape[0]} rows x {df.shape[1]} columns")

According to the original UCI codebook, `ca` should be 0-3 and `thal` should be 1-3. The `ca=4` and `thal=0` values we see are known encoding errors in this dataset. Since only a handful of rows are affected, we drop them rather than guessing at a replacement value — imputing arbitrarily here would just add noise.

### 4.4 Outliers

In [ ]:
num_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, col in zip(axes, num_cols):
    sns.boxplot(y=df[col], ax=ax, color='#4C72B0')
    ax.set_title(col)
plt.suptitle("Outlier check on the numeric variables (boxplots)", y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
for col in num_cols:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col:10s} -> {n_out} outliers (IQR method), bounds = [{lower:.1f}, {upper:.1f}]")

`chol` and `trestbps` have a few high values, but they're clinically plausible, cholesterol above 400 mg/dl does happen in very high-risk patients. We keep them. In a medical setting the extreme values are often the most informative for spotting disease, and cutting them would bias the model against exactly the sickest cases. We'll just keep this in mind when reading the results.

### 4.5 Useless columns

No useless columns to drop

### 4.6 Cleaning recap

In [1]:
print("Cleaning summary:")
print(f" - Rows before cleaning        : 1025")
print(f" - Duplicates removed          : 723")
print(f" - Final rows                  : {len(df)}")
print(f" - Final columns               : {df.shape[1]}")

Cleaning summary:
 - Rows before cleaning        : 1025
 - Duplicates removed          : 723


NameError: name 'df' is not defined

## 5. Exploratory Data Analysis

Now let's look at the target balance, some per-class statistics, and how the variables correlate.

### 5.1 Target balance

In [ ]:
target_counts = df['target'].value_counts()
print(target_counts)
print(f"\nClass 1 (disease): {target_counts[1]/len(df)*100:.1f}%")
print(f"Class 0 (healthy): {target_counts[0]/len(df)*100:.1f}%")

After cleaning the two classes are still fairly balanced (roughly 46/54), so it means accuracy is a reasonable metric here. We'll still report precision, recall and F1 to check if the dataset is really ok.

### 5.2 Per-class statistics

In [ ]:
df.groupby('target')[num_cols].mean().T

On average, patients with disease (`target=1`) show a higher maximum heart rate (`thalach`) and a lower `oldpeak` (ST depression) than healthy ones in this dataset. The `oldpeak` part lines up with the clinical literature, where a high value is a known risk factor

### 5.3 Correlation matrix

In [ ]:
plt.figure(figsize=(11, 9))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5)
plt.title("Correlation matrix across all variables", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
target_corr = corr['target'].drop('target').sort_values(key=abs, ascending=False)
print("Correlation of each variable with 'target' (sorted by strength):")
print(target_corr)

The variables most strongly correlated (in absolute value) with `target` are `cp` (chest pain type), `thalach`, `exang` (exercise-induced angina), `oldpeak`, `ca` and `thal`. No pair of features is extremely correlated with each other (nothing above 0.9), so multicollinearity isn't a real concern here.

## 6. Data Visualization

Each plot below comes with a title, axis labels, and a short read on what it's telling us.

### 6.1 Age distribution by disease status

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(data=df, x='age', hue='target', multiple='stack', bins=20, palette=['#55A868','#C44E52'])
plt.title("Age distribution by heart disease status")
plt.xlabel("Age (years)")
plt.ylabel("Number of patients")
plt.legend(title='target', labels=['Disease (1)', 'Healthy (0)'])
plt.tight_layout()
plt.show()

Disease shows up across a wide age range (peaking around 41-60), with no clean split by age alone. Age on its own won't separate the classe. We will need to combine several variables.

### 6.2 Chest pain type (cp) vs disease

In [ ]:
plt.figure(figsize=(7,5))
sns.countplot(data=df, x='cp', hue='target', palette=['#55A868','#C44E52'])
plt.title("Chest pain type (cp) by heart disease status")
plt.xlabel("Chest pain type (0-3)")
plt.ylabel("Number of patients")
plt.legend(title='target', labels=['Healthy (0)', 'Disease (1)'])
plt.tight_layout()
plt.show()

Chest pain type `cp=0` (typical angina) is strongly tied to *no* detected disease in this dataset, while types 1, 2 and 3 come with a much higher share of patients who do have disease. That matches the strong correlation we saw earlier and makes `cp` a key variable for the model.

### 6.3 Maximum heart rate (thalach) vs disease

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x='target', y='thalach', palette=['#55A868','#C44E52'])
plt.title("Maximum heart rate achieved by disease status")
plt.xlabel("target (0 = healthy, 1 = disease)")
plt.ylabel("thalach (bpm)")
plt.tight_layout()
plt.show()

Patients with disease have a higher median maximum heart rate than healthy ones, which looks counter-intuitive at first. It probably reflects a selection effect specific to this dataset (patients referred for a stress test) rather than a causal relationship we could generalize.

### 6.4 Sex vs disease

In [ ]:
plt.figure(figsize=(6,5))
ct = pd.crosstab(df['sex'], df['target'], normalize='index') * 100
ct.columns = ['Healthy (%)', 'Disease (%)']
ct.index = ['Female (0)', 'Male (1)']
ct.plot(kind='bar', stacked=True, color=['#55A868','#C44E52'], figsize=(6,5))
plt.title("Share of heart disease by sex")
plt.xlabel("Sex")
plt.ylabel("Share (%)")
plt.xticks(rotation=0)
plt.legend(title='')
plt.tight_layout()
plt.show()

In this dataset a larger share of female patients have heart disease than male patients. That likely reflects a clinical sampling bias (reason for referral, age of the women included) rather than a general epidemiological fact because we can read opposite facts on internet

## 7. Feature Engineering

Three things here:

1. **One-hot encoding** for the nominal categorical variables (`cp`, `restecg`, `slope`, `thal`), which have no natural ordering. this stops a linear model from reading a fake order into them.
2. A **derived `age_group`** feature (age bands), used only for exploration and *not* fed to the model, to avoid duplicating what `age` already provides.
3. **Standardization (StandardScaler)** of the continuous numeric variables, which helps the scale-sensitive models (Logistic Regression, KNN, SVM). We fit the scaler on the training set only, so no information leaks from test to train.

In [ ]:
df['age_group'] = pd.cut(df['age'], bins=[0,40,50,60,100],
                          labels=['<=40','41-50','51-60','60+'])
print(df.groupby('age_group')['target'].mean().rename('disease rate'))

The disease rate climbs with age up to 60 and then dips a little for the oldest group. This derived feature backs up what we saw in the EDA, but we leave it out of the model since it's redundant with `age`, which we keep as a continuous variable.

In [ ]:
categorical_nominal = ['cp', 'restecg', 'slope', 'thal']
df_model = pd.get_dummies(df.drop(columns=['age_group']), columns=categorical_nominal, drop_first=True)
print(f"Shape after one-hot encoding: {df_model.shape}")
df_model.head()

### 7.1 Features/target split and train-test split

In [ ]:
X = df_model.drop(columns=['target'])
y = df_model['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Train set: {X_train.shape[0]} patients")
print(f"Test set : {X_test.shape[0]} patients")
print(f"Class balance (train): {y_train.value_counts(normalize=True).round(2).to_dict()}")
print(f"Class balance (test) : {y_test.value_counts(normalize=True).round(2).to_dict()}")

### 7.2 Scaling (fit on train only)

In [ ]:
continuous_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test_scaled[continuous_cols] = scaler.transform(X_test[continuous_cols])

print("Scaling applied (mean/std computed on the training set only).")
X_train_scaled[continuous_cols].describe().T[['mean','std']]

## 8. Model Training

The brief asks for at least three models, so we train and compare four classifiers that suit this binary problem:

1. Logistic Regression
2. K-Nearest Neighbors (KNN)
3. Decision Tree
4. Random Forest

The scale-sensitive models (Logistic Regression, KNN) get the standardized data and the tree and forest ones use the raw data, since scaling doesn't matter for them.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
    'KNN (k=7)': KNeighborsClassifier(n_neighbors=7),
    'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_STATE, max_depth=5),
    'Random Forest': RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=200, max_depth=6),
}

scaled_models = {'Logistic Regression', 'KNN (k=7)'}

results = {}
predictions = {}
probabilities = {}

for name, model in models.items():
    Xtr = X_train_scaled if name in scaled_models else X_train
    Xte = X_test_scaled if name in scaled_models else X_test
    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)
    y_proba = model.predict_proba(Xte)[:, 1]

    predictions[name] = y_pred
    probabilities[name] = y_proba
    results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-score': f1_score(y_test, y_pred),
    }
    print(f"{name:22s} trained.")

### 8.1 Cross-validation (robustness check)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print("Mean accuracy from 5-fold cross-validation on the training set:\n")
for name, model in models.items():
    Xtr = X_train_scaled if name in scaled_models else X_train
    scores = cross_val_score(model, Xtr, y_train, cv=cv, scoring='accuracy')
    print(f"{name:22s} : {scores.mean():.3f} (+/- {scores.std():.3f})")

Cross-validation says the models are stable (small spread across folds) and roughly in line with the test-set scores, so no sign of overfitting

## 9. Model Evaluation

Since this is a reasonably balanced binary problem, we look at accuracy, precision, recall, F1, and a confusion matrix for each model.

### 9.1 Metrics comparison

In [ ]:
results_df = pd.DataFrame(results).T.round(3)
results_df = results_df.sort_values('F1-score', ascending=False)
results_df

In [ ]:
results_df.plot(kind='bar', figsize=(10,6), colormap='viridis')
plt.title("Model comparison across four metrics")
plt.ylabel("Score")
plt.xticks(rotation=20)
plt.ylim(0, 1.05)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

### 9.2 Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, (name, y_pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['Healthy','Disease'], yticklabels=['Healthy','Disease'])
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.suptitle("Confusion matrices for the four models", y=1.05)
plt.tight_layout()
plt.show()

### 9.3 Detailed classification report (best model)

In [ ]:
best_model_name = results_df.index[0]
print(f"Best model (by F1-score): {best_model_name}\n")
print(classification_report(y_test, predictions[best_model_name], target_names=['Healthy','Disease']))

## 10. Responsible AI Practices

**Data sensitivity.** This is medical data, which is sensitive by nature. The file is anonymized (no names, no patient IDs), but it's worth remembering that real health data always has to be handled with proper consent and under the relevant regulations (GDPR in Europe, HIPAA in the US).

**Bias and representativeness.** The dataset doesn't record the patients' precise geographic or ethnic background; it comes from the Cleveland database, so most likely a North American population from the late 1980s/early 1990s. A model trained on this could generalize poorly to other populations with different ages, backgrounds or lifestyles. We also saw a higher disease rate among women than men in this particular dataset, which runs against general epidemiological figures, probably a clinical sampling bias rather than a real medical truth, and definitely not something to read as one.

**Source reliability.** We found that this Kaggle dataset contains 723 exact duplicate rows (70% of the file), which isn't mentioned on the Kaggle page. A lot of public notebooks, including the one we used as a starting point, train on the data without removing them, which produces misleadingly good scores through data leakage. We chose to document this openly rather than quietly reproduce it.

**Honest model limitations.** Once the duplicates are gone, our models don't hit 90%+ accuracy, which is an honest result, not a failure to hide. An F1 around 0.80-0.85 on a test set of ~60 patients still carries a real margin of error. None of these models should be used as a standalone diagnostic.

**Misuse risks.** A model like this should never replace a professional medical diagnosis. A false negative (predicting "healthy" for a sick patient) could delay needed care; a false positive could cause anxiety and unnecessary tests. This project is educational and exploratory, not clinical.

**Use of generative AI tools.** *(fill in honestly as a group)*. For example: "An AI assistant helped scaffold the notebook and draft some of the explanations, but all the code was read, understood, run and checked by the group members." Be specific about how you actually used AI (brainstorming, debugging, proofreading, etc.), in line with the course's responsible-AI policy.

## 11. Conclusion and Future Work

**Main takeaways:**
- After proper cleaning (723 duplicates and a few invalid codes removed), we're left with 296 unique patients.
- The most discriminative variables for predicting heart disease are `cp` (chest pain type), `ca`, `thal`, `oldpeak` and `thalach`.
- Of the four models, **Logistic Regression** gives the best precision/recall/F1 trade-off on the test set (F1 ≈ 0.82, accuracy ≈ 0.80), a reminder that a simple, interpretable linear model can stay very competitive on a properly cleaned dataset like this.
- Unlike the reference Kaggle notebook (92% accuracy on the un-deduplicated data), our more modest but more trustworthy results show why careful cleaning matters before you model anything.

**What we learned:**
- Always check for duplicates and category-code consistency before modeling, even on a dataset that looks clean at first glance (zero missing values here).
- A high accuracy isn't automatically a good model; it can be hiding data leakage or an unhandled imbalance.
- It's worth comparing a few model families (linear, instance-based, tree, ensemble) instead of trusting a single algorithm.

**Limitations:**
- The cleaned dataset is fairly small (~300 patients), which limits statistical power and generalization.
- No precise demographic or geographic information about the patients.
- No exhaustive hyperparameter search (grid search) in this version; the values we used are reasonable but could be tuned further.

**Future work:**
- Run a hyperparameter search (GridSearchCV / RandomizedSearchCV) for each model.
- Try additional models (Gradient Boosting, XGBoost, kernel SVM) and ensembling techniques (stacking, voting).
- Validate on an independent external dataset to gauge real generalization.
- Explore stronger interpretability tools (SHAP values) to build more clinical trust in the predictions.